# Radiology Assistant: End-to-End Evaluation (Zip-Ready)

### Instructions for Colab:
1. **Upload your Zip**: Upload your project zip file to the Colab `/content` folder (using the folder icon on the left).
2. **Set Secrets**: Click the **🔑 (Secrets)** icon and add:
   - `OPENAI_API_KEY`, `HF_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY`.
3. **Run Cells**: The first code cell will automatically find and unzip your project.

In [ ]:
# 1. Unzip and Robust Path Setup
import os, sys, zipfile, shutil
from pathlib import Path

# Find the uploaded zip file in /content
zips = list(Path('/content').glob('*.zip'))
if not zips:
    print("No zip file found in /content. Assuming files are already present.")
    REPO_ROOT = Path.cwd()
else:
    zip_path = zips[0]
    extract_to = Path('/content/extracted_project')
    print(f"Unzipping {zip_path.name} to {extract_to}...")
    if extract_to.exists(): shutil.rmtree(extract_to)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    
    # Robustly find the directory containing 'src'
    # This handles nested folders or __MACOSX garbage
    src_dirs = list(extract_to.rglob('src'))
    if src_dirs:
        # The parent of the 'src' folder is our REPO_ROOT
        REPO_ROOT = src_dirs[0].parent
        print(f"Detected REPO_ROOT at: {REPO_ROOT}")
    else:
        REPO_ROOT = extract_to
        print(f"Warning: 'src' folder not found. Using {REPO_ROOT} as root.")

# CRITICAL: Add the 'src' directory to sys.path so 'bda_chest' can be found
SRC_PATH = str(REPO_ROOT / "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Python path updated. Searching for modules in: {SRC_PATH}")

# Verify bda_chest exists
if (REPO_ROOT / "src" / "bda_chest").exists():
    print("✅ Found 'bda_chest' package.")
else:
    print("❌ ERROR: 'bda_chest' package not found in expected location.")

# Install Dependencies
!pip install -q "transformers<5.0.0" "accelerate>=0.25" "bitsandbytes>=0.43.0" "evaluate>=0.4.0" "rouge_score>=0.1.2" kaggle python-dotenv openai Pillow timm

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# 2. Import Your Trained Pipeline
try:
    from bda_chest.pipeline import load_inference_bundle, infer_from_pil
    from bda_chest.llm import answer_question_about_report
    print("✅ Successfully imported project modules.")
except ImportError as e:
    print(f"❌ Import Error: {e}")
    print("Current sys.path:", sys.path)
    raise

# Find checkpoint
CHECKPOINT_PATH = REPO_ROOT / "eva_x_tiny_binary_best.pt"
if not CHECKPOINT_PATH.exists():
    # Try alternative locations if it's missing from root
    alt_path = list(REPO_ROOT.rglob("eva_x_tiny_binary_best.pt"))
    if alt_path:
        CHECKPOINT_PATH = alt_path[0]
    else:
        raise FileNotFoundError(f"Checkpoint not found at {CHECKPOINT_PATH}.")

bundle = load_inference_bundle(CHECKPOINT_PATH, device_hint="cuda")
print(f"Trained EVA-X model loaded from {CHECKPOINT_PATH}")

In [ ]:
# 3. Define Evaluation Judge (Med-Gemma)
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

class MedGemmaJudge:
    def __init__(self, model_id="google/medgemma-1.5-4b-it"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quant_config, device_map="auto")
        self.model.eval()

    def evaluate(self, q, a, c, gt):
        prompt = f"Evaluate the radiology assistant answer based on context. Return JSON with 'correctness_score' (1-5) and 'justification'.\n\nCONTEXT: {c}\nQ: {q}\nA: {a}\nGT: {gt}"
        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=300, temperature=0.1)
        return self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [ ]:
# 4. Download Kaggle Test Data
from kaggle.api.kaggle_api_extended import KaggleApi
import random

api = KaggleApi(); api.authenticate()
test_img_path = Path("/content/eval_test_images")
if not test_img_path.exists():
    api.dataset_download_files("paultimothymooney/chest-xray-pneumonia", path="/content/temp_kaggle", unzip=True)
    for cat in ["NORMAL", "PNEUMONIA"]:
        dest = test_img_path / cat.lower(); dest.mkdir(parents=True, exist_ok=True)
        imgs = list(Path("/content/temp_kaggle/chest_xray/test").joinpath(cat).glob("*.jpeg"))
        for i in random.sample(imgs, 3): shutil.copy(i, dest / i.name)
    shutil.rmtree("/content/temp_kaggle")
print("Test images ready at /content/eval_test_images")

In [ ]:
# 5. Run Evaluation Loop
from PIL import Image
judge = MedGemmaJudge()
test_files = list(test_img_path.rglob("*.jpeg"))

for path in test_files:
    img = Image.open(path).convert("RGB")
    q = "Are there signs of pneumonia?"
    
    # Stage 1: Classifier
    payload, prob = infer_from_pil(bundle, img, threshold=0.5)
    
    # Stage 2: QA Agent
    answer = answer_question_about_report(payload, q)
    
    # Stage 3: Feedback
    feedback = judge.evaluate(q, answer, payload['impression'], "Verify clinical accuracy.")
    
    print(f"\n--- Image: {path.name} ({path.parent.name}) ---")
    print(f"Impression: {payload['impression']}")
    print(f"Assistant: {answer}")
    print(f"Judge: {feedback}")